# Apt 305 — the wind-profile correction: terrain and height

**What changed.** Correction C2 replaces the ISO 13789 constant external convective
coefficient with $h_{ce} = 4 + 4u$. That correlation wants the wind **local to the
building surface**. The engine was feeding it the EPW wind column, which is a 10 m
reading over open terrain at the meteorological station — asserting, silently, that
Carlton and Essendon Fields aerodrome share both terrain class and measurement height.

**Why it matters.** The pivot at which $4u + 4 = 20\ \mathrm{W/(m^2K)}$ — the ISO constant —
is $u = 4$ m/s. Which side of that pivot the wind sits on decides the *sign* of C2, not
just its size.

This notebook reproduces the result set and shows the four things the task asked for:

| | |
| --- | --- |
| **Item 0** | what EnergyPlus was actually doing — measured, not assumed |
| **Item 1** | the ASHRAE two-layer profile, and the identity-case regression test |
| **Item 2** | **does C2 change sign?** |
| **Items 3 / 5** | sensitivity across terrain class, and three other Australian sites |

Nothing here is retyped. Every number is read from a committed result file, or
recomputed live from the engine, so the notebook and the repository cannot drift apart.

## 1 · Setup

In [ ]:
import os, subprocess, sys, json
from pathlib import Path

REPO   = Path('/content/AIB')
BRANCH = 'claude/new-session-carh9p'

if not REPO.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/samiraghafarigousheh-sys/aib.git', str(REPO)],
                   check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'pybuildingenergy/requirements.txt'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'matplotlib>=3.6', 'numpy', 'pandas', 'pytest'], check=True)

sys.path.insert(0, str(REPO / 'pybuildingenergy' / 'src'))
sys.path.insert(0, str(REPO / 'examples'))

print('branch ', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],
                                capture_output=True, text=True).stdout.strip())
print('commit ', subprocess.run(['git','rev-parse','--short','HEAD'],
                                capture_output=True, text=True).stdout.strip())

## 2 · The profile itself, computed live

$$u_{local} = u_{met}\left(\frac{\delta_{met}}{z_{met}}\right)^{a_{met}}\left(\frac{z}{\delta}\right)^{a}$$

Coefficients from **ASHRAE Handbook — Fundamentals (2021), Ch. 24, Table 1**. These are
also exactly what EnergyPlus derives from the `Terrain` field of its `Building` object,
which is what lets the two engines be driven by the same wind.

In [ ]:
from pybuildingenergy.source import utils as U
from apt305_building import build_bui

print(f"{'class':<14}{'a':>6}{'delta (m)':>11}   EnergyPlus Terrain")
print('-' * 58)
EP = {'flat_open': 'Ocean', 'open_country': 'Country',
      'suburban': 'Suburbs / Urban', 'city_centre': 'City'}
for cls, (a, d) in U._ASHRAE_TERRAIN_PARAMETERS.items():
    print(f'{cls:<14}{a:>6}{d:>11.0f}   {EP[cls]}')

print('\nThe identity case — the primary regression test:')
f_id = U.local_wind_speed_factor('open_country', 10.0)
print(f'  open_country at z = 10 m  ->  f = {f_id!r}   |f-1| = {abs(f_id-1):.3e}')
assert abs(f_id - 1.0) < 1e-12

print('\nApt 305, resolved from the building dictionary:')
factor, audit = U.resolve_local_wind_factor(build_bui())
for k, v in audit.items():
    print(f'  {k:<32} {v}')

In [ ]:
# The required input has no silent default -- that omission is the defect.
try:
    U.resolve_local_wind_factor({'building': {'height': 2.7}})
    raise SystemExit('NO GUARD -- a building with no terrain class was accepted')
except ValueError as exc:
    print('raises, as it must:\n')
    print(exc)

## 3 · Item 0 — what EnergyPlus was actually doing

Established two ways: statically, by reading the IDF and applying the documented
EnergyPlus algorithm; and dynamically, by re-running the committed IDF with
`Surface Outside Face Outdoor Air Wind Speed` reported hourly. The second is the
one that settles it — it is the wind EnergyPlus used, not the wind we think it used.

In [ ]:
wp = json.loads(Path('results/paper/wind_profile/wind_profile.json').read_text())
ep = wp['idf_static']

print('Static read of results/paper/validation_corrected/apt305_conditioned.idf')
print(f"  Site:HeightVariation   : {'present' if ep['site_heightvariation_present'] else 'ABSENT'}")
print(f"  Building Terrain field : {ep['terrain_field']}")
print(f"  -> a = {ep['exponent_a']}, delta = {ep['boundary_layer_delta_m']:.0f} m")
print()
for name, z in sorted(ep['wind_exposed_surface_heights_m'].items()):
    print(f"  {name:<20} z = {z:5.2f} m   f = {ep['factors'][name]:.4f}")

if wp['item0'].get('measured'):
    print('\nMeasured, by re-running that IDF with the wind reported hourly:')
    for k, v in sorted(wp['item0']['measured'].items()):
        print(f"  {k:<22} mean {v['mean_m_s']:.3f} m/s over {v['n']:,} h")

print('\n' + '=' * 72)
print(wp['item0']['verdict'].replace('**', ''))
print('=' * 72)

## 4 · Item 2 — does C2 change sign?

The controlled experiment: **the same engine, run twice, changing only the $h_{ce}$
model.** Run once on the raw station column and once on the terrain-corrected wind.

Both "before" and "after" are produced by *this* engine tree. An older `wind_stats.json`
would also carry every closure fix made since, and would attribute those to the terrain
correction.

In [ ]:
st = json.loads(Path('results/paper/wind_profile/wind_stats_station.json').read_text())['summary']
tr = json.loads(Path('results/paper/wind_profile/wind_stats_terrain.json').read_text())['summary']

rows = [
    ('Wind fed to h_ce',            'station, 10 m',                 f"local, {tr['wind_audit']['terrain_class']} @ {tr['wind_audit']['surface_height_m']:.2f} m"),
    ('Annual mean wind (m/s)',      f"{st['mean_wind_annual']:.2f}", f"{tr['mean_wind_annual']:.2f}"),
    ('Hours above the 4 m/s pivot', f"{st['pct_hours_above_pivot']:.1f} %", f"{tr['pct_hours_above_pivot']:.1f} %"),
    ('Mean h_ce W/(m2 K)',          f"{st['mean_h_ce_annual']:.2f}", f"{tr['mean_h_ce_annual']:.2f}"),
    ('', '', ''),
    ('Sensible cooling, ISO fixed', f"{st['C_fix']:.2f}",            f"{tr['C_fix']:.2f}"),
    ('Sensible cooling, 4u + 4',    f"{st['C_dyn']:.2f}",            f"{tr['C_dyn']:.2f}"),
    ('C2 effect on cooling (kWh)',  f"{st['delta_C']:+.2f}",         f"{tr['delta_C']:+.2f}"),
    ('C2 effect on heating (kWh)',  f"{st['delta_H']:+.2f}",         f"{tr['delta_H']:+.2f}"),
]
print(f"{'':<30}{'station wind':>26}{'terrain-corrected':>28}")
print('-' * 84)
for a, b, c in rows:
    print(f'{a:<30}{b:>26}{c:>28}')
print('-' * 84)

# The ISO-fixed column must be identical: with h_ce on 'table' the wind is never
# consumed. If it moved, something other than the wind changed between the runs.
assert abs(st['C_fix'] - tr['C_fix']) < 1e-9, 'the control arm moved -- the experiment is not controlled'
print('control arm identical to 1e-9: the only thing that differs is the wind.\n')

flip = (st['delta_C'] < 0) != (tr['delta_C'] < 0)
print(('C2 REVERSES SIGN on sensible cooling: '
       f"{st['delta_C']:+.2f} kWh -> {tr['delta_C']:+.2f} kWh") if flip else
      f"C2 does not reverse: {st['delta_C']:+.2f} -> {tr['delta_C']:+.2f} kWh")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.5, 5.0))

# A -- the two wind distributions against the pivot, which is the whole argument
epw = Path('weather_cache/AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw')
ws = np.array([float(l.split(',')[21]) for i, l in enumerate(epw.read_text(errors='ignore').splitlines())
               if i >= 8 and len(l.split(',')) > 21])
wl = ws * tr['wind_factor']
bins = np.linspace(0, ws.max(), 46)
axA.hist(ws, bins=bins, color='#8A8A8A', alpha=0.45, label=f'station  (mean {ws.mean():.2f} m/s)')
axA.hist(wl, bins=bins, color='#2a78d6', alpha=0.80, label=f'local    (mean {wl.mean():.2f} m/s)')
axA.axvline(4.0, color='#c2255c', ls='--', lw=1.8)
axA.text(4.15, axA.get_ylim()[1]*0.92, 'pivot 4 m/s\n$4u+4 = 20$ = ISO fixed',
         color='#c2255c', fontsize=8.5, va='top', style='italic')
axA.set_xlabel('wind speed  [m/s]'); axA.set_ylabel('hours')
axA.set_title(f"Above the pivot: station {st['pct_hours_above_pivot']:.1f} %  "
              f"->  local {tr['pct_hours_above_pivot']:.1f} %", fontsize=11)
axA.legend(frameon=False, fontsize=9)

# B -- the controlled experiment, both ways
x = np.arange(2); w = 0.35
axB.bar(x - w/2, [st['C_fix'], st['C_dyn']], w, color='#8A8A8A', label='station wind')
axB.bar(x + w/2, [tr['C_fix'], tr['C_dyn']], w, color='#2a78d6', label='terrain-corrected')
for xi, vals in zip(x, ([st['C_fix'], tr['C_fix']], [st['C_dyn'], tr['C_dyn']])):
    for dx, v in zip((-w/2, w/2), vals):
        axB.annotate(f'{v:.2f}', (xi + dx, v), xytext=(0, 3), textcoords='offset points',
                     ha='center', fontsize=9, fontweight='bold')
axB.set_xticks(x); axB.set_xticklabels(['ISO fixed\n$h_{ce}$ = 20', 'Dynamic\n$h_{ce}=4u+4$'])
axB.set_ylabel('Annual sensible cooling (kWh)')
axB.set_title(f"C2 effect: {st['delta_C']:+.2f} kWh  ->  {tr['delta_C']:+.2f} kWh", fontsize=11)
axB.legend(frameon=False, fontsize=9)
for ax in (axA, axB):
    ax.grid(axis='y', alpha=0.3, ls='--'); ax.set_axisbelow(True)
    for s_ in ('top', 'right'): ax.spines[s_].set_visible(False)

fig.tight_layout(); plt.show()

## 5 · Items 3 and 5 — terrain sensitivity, and other Australian sites

Terrain class is a judgement the modeller assigns by inspection, not a measurement, so
the effect of each of the four choices is reported rather than buried in one line.

In [ ]:
print(f"Item 3 — the four terrain classes at Apt 305's z = {wp['site']['surface_height_m']:.2f} m\n")
print(f"{'class':<14}{'a':>6}{'delta':>8}{'factor':>9}{'mean u':>9}{'>pivot':>9}{'mean h_ce':>11}")
print('-' * 66)
for r in wp['terrain_rows']:
    mark = ' <-' if r['terrain_class'] == wp['site']['terrain_class'] else ''
    print(f"{r['terrain_class']:<14}{r['exponent_a']:>6}{r['boundary_layer_delta_m']:>8.0f}"
          f"{r['factor']:>9.4f}{r['mean_local_m_s']:>9.2f}"
          f"{r['pct_hours_above_pivot']:>8.1f}%{r['mean_h_ce']:>11.2f}{mark}")
print(f"{'station':<14}{'-':>6}{'-':>8}{1.0:>9.4f}{wp['epw']['mean_m_s']:>9.2f}"
      f"{wp['epw']['pct_above_pivot']:>8.1f}%{wp['epw']['mean_h_ce']:>11.2f}")
print('\nEvery class puts the mean h_ce BELOW the ISO constant of 20; the unadjusted')
print('station wind puts it above. The sign of C2 is robust to the terrain judgement')
print('even though its magnitude is not.')

In [ ]:
print('Item 5 — the engine must serve sites beyond this one\n')
print(f"{'case':<44}{'terrain':<14}{'z (m)':>7}{'factor':>9}{'mean u':>9}")
print('-' * 83)
for r in wp['au_cases']:
    print(f"{r['case']:<44}{r['terrain_class']:<14}{r['surface_height_m']:>7.2f}"
          f"{r['factor']:>9.4f}{r['mean_local_m_s']:>9.2f}")

print('\nThe height at which each class crosses f = 1.0 (the building starts seeing')
print('MORE wind than the 10 m station):\n')
for cls, z in wp['crossings']:
    print(f'  {cls:<14} {z:7.1f} m   (~level {max(1, round(z/3)):.0f} at 3 m storeys)')
print('\nNote the last row. The task expected a Level-20 city-centre apartment to come')
print(f"out ABOVE 1.0; it does not — the factor is {U.local_wind_speed_factor('city_centre', 58.0):.4f}. The same")
print('roughness that slows the wind near the ground also thickens the boundary layer')
print('that has to be climbed, so city_centre does not cross until ~114 m. The check the')
print('task wanted still passes, on a different row: suburban at 58 m gives '
      f"{U.local_wind_speed_factor('suburban', 58.0):.4f}.")

## 6 · The result set, before and after

The canonical trajectory gains a fourteenth state. The thirteen before it are unchanged —
each is a cherry-pick of one historical commit, and none of them contains the wind
profile — so the whole of the movement is in the last row.

In [ ]:
import pandas as pd

new = pd.read_csv('results/paper/trajectory_v2/comparison.csv')
old = pd.read_csv('results/paper_pre_wind_profile/trajectory_v2/comparison.csv')
cols = ['State', 'Sensible heating (kWh)', 'Sensible cooling (kWh)',
        'Total (kWh/m²)', 'V2 residual (%)', '< 5 %?']
print('AFTER — results/paper/trajectory_v2/comparison.csv')
print(new[[c for c in cols if c in new.columns]].to_string(index=False))
print('\nBEFORE — results/paper_pre_wind_profile/trajectory_v2/comparison.csv (last row)')
print(old[[c for c in cols if c in old.columns]].tail(1).to_string(index=False))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/paper/SUPERSEDED_wind_profile.md').read_text()))

## 7 · The gate

No headline is final unless the V2 residual, the transmission inventory, the latent gate
and the regression suite all pass. Run them.

In [ ]:
raw = json.loads(Path('results/paper/trajectory_v2/trajectory_raw.json').read_text())

worst = max(abs(v['residual_pct']) for v in raw['results'].values()
            if v.get('residual_pct') is not None)
items = {k: v['sankey']['n_transmission_items'] for k, v in raw['results'].items()}
final = raw['results'][list(raw['results'])[-1]]

print(f"V2 residual, worst of all states : {worst:.4f} %   {'PASS' if worst < 5 else 'FAIL'} (< 5 %)")
print(f"Transmission line items          : {sorted(set(items.values()))}   "
      f"{'PASS' if set(items.values()) == {7} else 'FAIL'} (7 expected)")
print(f"Latent heating, final state      : {final['Q_H_latent_kWh']:.4f} kWh   "
      f"{'PASS' if abs(final['Q_H_latent_kWh']) < 0.05 else 'FAIL'}")
print(f"Engine tree vs HEAD              : "
      f"{'IDENTICAL — PASS' if raw['engine_tree_check']['identical'] else 'DIFFERS — FAIL'}")
print(f"Final state vs a live HEAD run   : "
      f"{'PASS' if raw['canonical_check'].get('ok', raw['canonical_check'].get('within_tolerance')) else 'FAIL'}")

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-q'],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
print('exit code:', r.returncode)

## 8 · Reproduce it from scratch

Each command is independent and writes only its own outputs. EnergyPlus is needed for
the first two only.

```bash
# EnergyPlus 24.1, for Item 0 and the matched validation
wget -qO /tmp/ep.tar.gz https://github.com/NREL/EnergyPlus/releases/download/v24.1.0/EnergyPlus-24.1.0-9d7789a3ac-Linux-Ubuntu22.04-x86_64.tar.gz
mkdir -p /opt/ep && tar -xzf /tmp/ep.tar.gz -C /opt/ep --strip-components=1

W=weather_cache/AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw

# Items 0, 1, 3, 5 -- the profile, and what EnergyPlus was doing
python tools/diagnostics/wind_profile_terrain.py --weather $W \
    --outdir results/paper/wind_profile --energyplus /opt/ep/energyplus

# Item 2 -- the C2 sign, both ways. The station run comes first; it is the 'before'.
python tools/diagnostics/wind_h_ce_diagnostic.py --weather $W \
    --outdir results/paper/wind_profile --tag station --no-wind-profile
python tools/diagnostics/wind_h_ce_diagnostic.py --weather $W \
    --outdir results/paper/wind_profile --tag terrain \
    --compare-to results/paper/wind_profile/wind_stats_station.json

# Item 4 -- the gated set. --closure-ref pins the instrument so the wind-profile
# commit is not swept into it; the harness aborts if it is.
python tools/diagnostics/canonical_trajectory.py --weather $W \
    --outdir results/paper/trajectory_v2 --closure-ref 0935730 --closure-base 978db37
python examples/corrected_vs_energyplus.py --energyplus /opt/ep/energyplus
python tools/figures/make_all_figures.py
python -m pytest tests/ -q
```

In [ ]:
# The figures, rebuilt and shown.
r = subprocess.run([sys.executable, 'tools/figures/make_all_figures.py'],
                   capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode != 0:
    print('--- stderr ---\n', r.stderr[-2500:])

from IPython.display import Image
FIGDIR = REPO / 'results' / 'paper' / 'figures'
for stem in ('F6_wind_field_and_c2', 'F3_correction_trajectory',
             'F4_per_correction_waterfall'):
    p = FIGDIR / f'{stem}.png'
    if p.exists():
        display(Markdown(f'### `{p.name}`'))
        display(Image(filename=str(p), width=1000))

In [ ]:
import shutil
archive = shutil.make_archive('/content/AIB_wind_profile', 'zip',
                              root_dir=str(REPO / 'results' / 'paper'))
print(archive, f'({Path(archive).stat().st_size:,} B)')
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('not running in Colab -- zip left on disk')